# QuickTalk: HellaSwag score (run 8)

Common-sense test: 10,042 everyday situations with 4 possible endings; the model picks the most likely ending
(random guessing = 25%). **Run this only after run 8 has finished**, in a new runtime (any GPU; A100 ~10 min, T4 ~40 min).
It only reads the finished checkpoints and never changes the training folder.

Published reference (acc_norm, approximate): GPT-2 small 124M ~30%, Pythia-160M ~30%, MobileLLM-125M ~39%,
SmolLM2-135M ~42%. Cell 4 also scores SmolLM2-135M (base) with the same scorer for a like-for-like comparison.

In [ ]:
# 1. Drive and code (own folder /content/qt_eval, separate from the training notebook's /content/quicktalk)
import os, subprocess
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
RUN = 'quicktalk_run8'; WORK = f'/content/drive/MyDrive/{RUN}'; SRC = '/content/qt_eval'
assert os.path.exists(f'{WORK}/checkpoints/pretrain_final.pt'), 'run 8 pretraining has not finished yet'
if not os.path.exists(SRC):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'claude/intelligent-ride-oxqjfy', '--filter=blob:none',
                    '--sparse', 'https://github.com/sraivante/quicktalk.git', SRC], check=True)
    subprocess.run(['git', '-C', SRC, 'sparse-checkout', 'set', 'train/colab'], check=True)
HS = f'{SRC}/train/colab/hellaswag_eval.py'
!pip -q install datasets tokenizers

In [ ]:
# 2. Run 8 base model (after pretraining): the number to compare with published small models
!python {HS} --workdir {WORK} --ckpt {WORK}/checkpoints/pretrain_final.pt --out {WORK}/hellaswag_pretrain.json

In [ ]:
# 3. Run 8 chat model (after chat training), for reference
import os
if os.path.exists(f'{WORK}/checkpoints/sft_final.pt'):
    !python {HS} --workdir {WORK} --ckpt {WORK}/checkpoints/sft_final.pt --out {WORK}/hellaswag_sft.json

In [ ]:
# 4. Optional: SmolLM2-135M base with the same scorer (like-for-like comparison)
!pip -q install transformers
!python {HS} --hf-model HuggingFaceTB/SmolLM2-135M --out {WORK}/hellaswag_smollm2_135m.json

In [ ]:
# 5. Summary
import json, glob
for p in sorted(glob.glob(f'{WORK}/hellaswag_*.json')):
    r = json.load(open(p)); print(f"{p.split('/')[-1]:32} acc_norm {r['acc_norm']:.1%}  acc {r['acc']:.1%}  ({r['items']} items)")